**`enrich_footprints_with_raster`**

First implementation:

Sample the HISDAC first-build-up-year (FBUY) raster at harmonized footprint centroids
and write `year_first_buildup_hisdac` to a per-county sidecar parquet.

# Configure

In [ ]:
import argparse

import numpy as np
import rasterio

from openplaces.io import save_parquet
from openplaces.io.readers import get_dataset, get_entities
from openplaces.recipe import get_output_path, get_recipe_by_id
from openplaces.utils import pretty_print

In [ ]:
parser = argparse.ArgumentParser(
    description='Enrich harmonized footprints with HISDAC first-build-up-year'
)
parser.add_argument(
    '--entity_recipe_id',
    help='Harmonized footprint recipe (e.g. "US_footprint-cheer-2026")',
)
parser.add_argument(
    '--dataset_recipe_id',
    help='FBUY raster recipe (e.g. "US_built-year-hisdac-v1")',
)
parser.add_argument(
    '--admin_ids',
    help='Admin unit IDs to enrich (e.g. "US-FL-AL")',
    nargs='*',
)
parser.add_argument(
    '--nodata',
    help='Raster nodata value to replace with NaN (default: 0)',
    type=float,
    default=0,
)
parser.add_argument(
    '--reprocess',
    help='Reprocess admin IDs even if output already exists',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    '--entity_recipe_id US_footprint-spine-2026 '
    '--dataset_recipe_id US_built-year-hisdac-v1 '
    '--admin_ids US-NC-BR '
    # '--admin_ids US-FL-AL '
    '--reprocess '
    '--verbose'
)

args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

In [ ]:
# Show recipe parameters
pretty_print(get_recipe_by_id(args.entity_recipe_id))

In [ ]:
pretty_print(get_recipe_by_id(args.dataset_recipe_id))

# Enrich

In [ ]:
entity_recipe = get_recipe_by_id(args.entity_recipe_id)

for admin_id_str in args.admin_ids:
    base_path = get_output_path(entity_recipe, admin_id_str)
    out_path = base_path.with_stem(base_path.stem + f'_{args.dataset_recipe_id}')

    if out_path.exists() and not args.reprocess:
        if args.verbose:
            print(f'{admin_id_str}: output exists, skipping')
        continue

    raster_path = get_dataset(args.dataset_recipe_id, admin_id_str)
    if raster_path is None or not raster_path.exists():
        print(f'{admin_id_str}: raster not found, skipping')
        continue

    footprints = get_entities(args.entity_recipe_id, admin_id_str, geom=True)

    centroids = footprints.geometry.centroid
    with rasterio.open(raster_path) as src:
        centroids_proj = centroids.to_crs(src.crs.wkt)
        coords = list(zip(centroids_proj.x, centroids_proj.y))
        values = np.array([v[0] for v in src.sample(coords)], dtype=float)

    if args.nodata is not None:
        values[values == args.nodata] = np.nan

    footprints['year_first_buildup_hisdac'] = values

    out_path.parent.mkdir(parents=True, exist_ok=True)
    save_parquet(footprints, out_path)

    if args.verbose:
        n_valid = int((~np.isnan(values)).sum())
        print(f'{admin_id_str}: {n_valid}/{len(footprints)} valid')

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/...'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect results

In [ ]:
import pandas as pd

footprints = pd.read_parquet(out_path)

In [ ]:
footprints['n_parcels_per_footprint'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
mask = footprints['year_built_parcel'].fillna(0).gt(1950) & footprints[
    'year_first_buildup_hisdac'
].fillna(0).gt(1950)
_, _, _, img = ax.hist2d(
    footprints[mask]['year_built_parcel'],
    footprints[mask]['year_first_buildup_hisdac'],
    bins=np.arange(1952.5, 2027.5, 5),
    cmap='Blues',
)
plt.colorbar(img)
ax.plot([1950, 2030], [1950, 2030])
ax.set_ylabel('HISDAC: first built year')
ax.set_xlabel('Parcel: built year')